In this project , we are predicting customer churn for a telecom company.Churn is when a customer stops using a service. A telecom company has many customers, and every month some of them decide to terminate their contracts and switch to another provider. Losing customers is expensive, so the company wants to identify the people who are likely to leave before they actually do.
The idea is simple: for every customer, we compute a score between 0 and 1 - the probability that this customer is about to churn. Then the company can target the customers with high scores and try to retain them, for example by sending an email with discounts or special promotions.

To predict this score we use machine learning. The task is binary classification: we look at one customer and predict one of two possible outcomes. For the i-th customer, we want our model g to approximate the target y:

g(xi) ≈ yi

In the formula, yi is the model's prediction and belongs to {0,1}, with 0 being the negative value or no churning, and 1 the positive value or churning. The output corresponds to the likelihood of churning.

In brief, the main idea behind this project is to build a model with historical data from customers and assign a score of the likelihood of churning.

In [11]:
import pandas as pd # To import pandas library for data manipulation and analysis
import numpy as np # To import numpy library for numerical computations

import matplotlib.pyplot as plt # To import matplotlib library for data visualization
import seaborn as sns # To import seaborn library for statistical data visualization

# To display plots inline in Jupyter Notebook
%matplotlib inline 

Dataset used in this project is donwloded from Kaggle , URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

In [12]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv') # To read the CSV file containing customer churn data into a pandas DataFrame
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [13]:
df.head().T # To show the first 5 rows of the DataFrame and transpose it for better readability, which converts rows to columns and columns to rows. This is useful for quickly inspecting the data and understanding its structure.

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [14]:
df.columns = df.columns.str.lower().str.replace(' ', '_') # To convert all column names to lowercase and replace spaces with underscores for consistency and easier access

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index) # To create a list of column names that have data type 'object', which typically indicates categorical variables

for col in categorical_columns: # To iterate over each column name in the list of categorical columns
    df[col] = df[col].str.lower().str.replace(' ', '_') # To convert all string values in the column to lowercase and replace spaces with underscores for consistency and easier analysis
    
df

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-vhveg,female,0,yes,no,1,no,no_phone_service,dsl,no,...,no,no,no,no,month-to-month,yes,electronic_check,29.85,29.85,no
1,5575-gnvde,male,0,no,no,34,yes,no,dsl,yes,...,yes,no,no,no,one_year,no,mailed_check,56.95,1889.5,no
2,3668-qpybk,male,0,no,no,2,yes,no,dsl,yes,...,no,no,no,no,month-to-month,yes,mailed_check,53.85,108.15,yes
3,7795-cfocw,male,0,no,no,45,no,no_phone_service,dsl,yes,...,yes,yes,no,no,one_year,no,bank_transfer_(automatic),42.30,1840.75,no
4,9237-hqitu,female,0,no,no,2,yes,no,fiber_optic,no,...,no,no,no,no,month-to-month,yes,electronic_check,70.70,151.65,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-resvb,male,0,yes,yes,24,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,yes,mailed_check,84.80,1990.5,no
7039,2234-xaduh,female,0,yes,yes,72,yes,yes,fiber_optic,no,...,yes,no,yes,yes,one_year,yes,credit_card_(automatic),103.20,7362.9,no
7040,4801-jzazl,female,0,yes,yes,11,no,no_phone_service,dsl,yes,...,no,no,no,no,month-to-month,yes,electronic_check,29.60,346.45,no
7041,8361-ltmkd,male,1,yes,no,4,yes,yes,fiber_optic,no,...,no,no,no,no,month-to-month,yes,mailed_check,74.40,306.6,yes


In [ ]:
df.dtypes
# We notice some columns like 'totalcharges' are of object type, which should be numeric. We will convert them to numeric type and handle any errors that arise during the conversion.
# And seniorcitizen column is of int type, we will convert it to object type for consistency with other categorical variables.


customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [16]:
tc = pd.to_numeric(df.totalcharges, errors='coerce') # To convert the 'totalcharges' column to numeric type, coercing any errors (e.g., non-numeric values) to NaN
df.totalcharges = tc # To replace the original 'totalcharges' column with the converted numeric column

In [19]:
df.totalcharges = df.totalcharges.fillna(0) # To fill any NaN values in the 'totalcharges' column with 0, ensuring that there are no missing values in this numeric column

In [20]:
df[tc.isnull()][['customerid' , 'totalcharges']] # To display the rows where 'totalcharges' could not be converted to numeric (i.e., where it is NaN), showing the 'customerid' and 'totalcharges' columns for reference


,customerid,totalcharges
488,4472-lvygi,0.0
753,3115-czmzd,0.0
936,5709-lvoeq,0.0
1082,4367-nuyao,0.0
1340,1371-dwpaz,0.0
3331,7644-omvmy,0.0
3826,3213-vvolg,0.0
4380,2520-sgtta,0.0
5218,2923-arzlg,0.0
6670,4075-wkniu,0.0


In [25]:
df.churn = (df.churn == 'yes').astype(int).sum() # To count the number of customers who have churned by converting the 'churn' column to binary (1 for 'yes', 0 for 'no') and summing the values
df.churn.head()


0    0
1    0
2    0
3    0
4    0
Name: churn, dtype: int64